In [3]:
# Instalamos la dependencia de Yahoo Finance en el cuaderno
!pip install -q yfinance numba plotly nbformat numpy matplotlib statsmodels scikit-learn tensorflow

# Importamos la función de nuestro nuevo archivo local
from data_prep import descargar_y_preparar_datos

In [4]:
# Ejecutamos la descarga y preparación para EURJPY
df_hist, df_test, df_raw = descargar_y_preparar_datos("EURJPY=X")

=== INICIANDO DESCARGA DE DATOS PARA EURJPY=X ===


[*********************100%***********************]  1 of 1 completed

-> Registros descargados de Yahoo Finance: 6151
-> Datos de Entrenamiento (Semanas Completas): 6007 velas
-> Datos de Forward Test (Semana Actual): 101 velas
=== PROCESO DE ADQUISICIÓN COMPLETADO EXITOSAMENTE ===
Archivos listos en el disco local: 
1. EURJPY_H1_Hist_Semanas_Completas.csv 
2. EURJPY_H1_Semana_Actual_Prueba.csv 
3. EURJPY_D1_ultimo_ano.csv


---

# Trading Systems & Backtesting

Esta sección final se encarga de convertir las predicciones probabilísticas y estadísticas de los modelos en decisiones de inversión ejecutables. Implementa una estrategia operativa basada en el análisis de gradientes y límites dinámicos de volatilidad, simulando el comportamiento de un bot de trading en condiciones operativas reales para evaluar su viabilidad financiera.

---

## 1. Generación de Señales y Triggers Operativos
Este proceso traduce el consenso predictivo de la red neuronal y los canales de varianza en gatillos de compra o venta en tiempo real para la semana de prueba:
*   **Gradiente de Tendencia (Derivada):** Analiza la pendiente de la predicción central para detectar puntos de inflexión temporal (giros alcistas y bajistas).
*   **Reversión a la Media (Ruptura de Varianza):** Dispara operaciones de compra o venta cuando el precio de mercado real perfora los límites del canal gaussiano ($\pm 2\sigma$), asumiendo un agotamiento estadístico del precio.
*   **Posicionamiento de Señales:** Coloca visualmente indicadores geométricos (triángulos verdes de compra y rojos de venta) en los puntos de quiebre temporal de las velas japonesas.

---

## 2. Simulador de Trading y Curva de Capital
El motor de simulación evalúa la tasa de acierto y el rendimiento monetario neto de las señales bajo una gestión de riesgo estricta:
*   **Gestión de Capital y Payout:** Configura un capital inicial base de $100 USD, un tamaño de posición estático de $10 USD y un retorno de ganancia (*payout*) del 82% típico de brokers de opciones binarias.
*   **Resolución de Trades:** Evalúa el resultado de cada señal comparando el precio de entrada contra el precio de expiración exacta a 1 hora (siguiente vela horaria), contabilizando el balance neto acumulado.
*   **Gráfico de Doble Panel:** Integra en un único entorno la visualización técnica del mercado (velas, consenso y señales) y la curva de crecimiento de la equidad (*equity curve*), reportando la tasa de acierto (*Win Rate*) final.

---

## Resumen de Parámetros y Resultados Operativos

| Parámetro / Métrica | Valor Estándar | Propósito Técnico | Tipo de Métrica |
| :--- | :--- | :--- | :--- |
| **Sensibilidad de Gradiente** | `0.01` | Umbral para filtrar pequeños ruidos de la predicción central. | Configuración de Entrada |
| **Volumen por Operación** | `$10.00 USD` | Tamaño estático de la posición para cada trigger disparado. | Gestión de Riesgo |
| **Expiración de Trade** | `1 Hora (H1)` | Tiempo de permanencia en el mercado antes de liquidar el balance. | Regla del Sistema |
| **Win Rate de Control** | Dinámico (%) | Porcentaje de operaciones ganadas sobre el total de trades ejecutados. | Rendimiento Estadístico |
| **Balance de Capital** | Dinámico ($) | Evolución de la equidad acumulada a lo largo de la simulación. | Rendimiento Financiero |

In [2]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

symb = "EURJPY"
print(f"=== INICIANDO SISTEMA FINANCIERO Y BACKTESTING PARA {symb} ===")

# 1. Cargamos directamente la hoja de pronósticos generada por la IA
df_velas_forward = pd.read_csv(f"{symb}_H1_Predictions_Blind_Week.csv")
df_velas_forward['time'] = pd.to_datetime(df_velas_forward['time'])

# 2. Extraemos los vectores para el simulador
fechas_forward = df_velas_forward['time'].values
prediccion_central = df_velas_forward['prediccion_central'].values
banda_superior = df_velas_forward['banda_superior'].values
banda_inferior = df_velas_forward['banda_inferior'].values

print(f"-> Pronósticos cargados exitosamente. {len(df_velas_forward)} horas listas para operar.")

=== INICIANDO SISTEMA FINANCIERO Y BACKTESTING PARA EURJPY ===
-> Pronósticos cargados exitosamente. 101 horas listas para operar.


In [3]:
# =====================================================================
# ALGORITMO OPERATIVO Y SIMULADOR DE TRADING (BACKTESTING)
# =====================================================================
print("-> Ejecutando Motor de Backtesting...")

# Parámetros de Control (Ajustables)
sensibilidad_gradiente = 0.01
capital = 100.0
apuesta_por_trade = 10.0
payout = 0.82

precios_ciegos = df_velas_forward['close'].values
gradiente_pred = np.gradient(prediccion_central)
señales_compra_x, señales_compra_y, señales_venta_x, señales_venta_y = [], [], [], []
trades_ganados, trades_perdidos = 0, 0
historial_capital, fechas_capital = [capital], [fechas_forward[0]]

# Escaneo del futuro predictivo
for i in range(1, len(fechas_forward) - 1):
    giro_alcista = (gradiente_pred[i-1] < -sensibilidad_gradiente) and (gradiente_pred[i] > sensibilidad_gradiente)
    giro_bajista = (gradiente_pred[i-1] > sensibilidad_gradiente) and (gradiente_pred[i] < -sensibilidad_gradiente)
    rompe_banda_inf = precios_ciegos[i] < banda_inferior[i]
    rompe_banda_sup = precios_ciegos[i] > banda_superior[i]

    precio_entrada = precios_ciegos[i]
    precio_expiracion = precios_ciegos[i+1]

    if giro_alcista or rompe_banda_inf:
        señales_compra_x.append(fechas_forward[i])
        señales_compra_y.append(df_velas_forward['low'].iloc[i] - 0.08)
        if precio_expiracion > precio_entrada:
            capital += (apuesta_por_trade * payout); trades_ganados += 1
        else:
            capital -= apuesta_por_trade; trades_perdidos += 1
        historial_capital.append(capital); fechas_capital.append(fechas_forward[i+1])

    elif giro_bajista or rompe_banda_sup:
        señales_venta_x.append(fechas_forward[i])
        señales_venta_y.append(df_velas_forward['high'].iloc[i] + 0.08)
        if precio_expiracion < precio_entrada:
            capital += (apuesta_por_trade * payout); trades_ganados += 1
        else:
            capital -= apuesta_por_trade; trades_perdidos += 1
        historial_capital.append(capital); fechas_capital.append(fechas_forward[i+1])

total_trades = trades_ganados + trades_perdidos
win_rate = (trades_ganados / total_trades * 100) if total_trades > 0 else 0
print(f"[+] Backtest Finalizado | Trades: {total_trades} | Win Rate: {win_rate:.1f}% | Capital Final: ${capital:.2f}")

# =====================================================================
# RENDERIZADO DEL DASHBOARD OPERATIVO DUAL
# =====================================================================
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08, row_heights=[0.7, 0.3],
                    subplot_titles=(f'Triggers de Gradiente (Sens. {sensibilidad_gradiente})', f'Curva de Capital | Final: ${capital:.2f}'))

fig.add_trace(go.Scatter(x=pd.concat([pd.Series(fechas_forward), pd.Series(fechas_forward)[::-1]]),
                         y=np.concatenate([banda_superior, banda_inferior[::-1]]),
                         fill='toself', fillcolor='rgba(41,128,185,0.15)', line=dict(color='rgba(255,255,255,0)'), name='Canal Predictivo'), row=1, col=1)
fig.add_trace(go.Candlestick(x=df_velas_forward['time'], open=df_velas_forward['open'], high=df_velas_forward['high'],
                             low=df_velas_forward['low'], close=df_velas_forward['close'], name='Mercado',
                             increasing_line_color='rgba(39,174,96,1)', decreasing_line_color='rgba(231,76,60,1)'), row=1, col=1)
fig.add_trace(go.Scatter(x=fechas_forward, y=prediccion_central, mode='lines', name='Consenso', line=dict(color='darkblue', width=3)), row=1, col=1)
fig.add_trace(go.Scatter(x=señales_compra_x, y=señales_compra_y, mode='markers', name='COMPRA',
                         marker=dict(symbol='triangle-up', size=14, color='rgba(46,204,113,0.9)', line=dict(width=2, color='darkgreen'))), row=1, col=1)
fig.add_trace(go.Scatter(x=señales_venta_x, y=señales_venta_y, mode='markers', name='VENTA',
                         marker=dict(symbol='triangle-down', size=14, color='rgba(231,76,60,0.9)', line=dict(width=2, color='darkred'))), row=1, col=1)

color_capital = 'green' if capital >= 100.0 else 'red'
fig.add_trace(go.Scatter(x=fechas_capital, y=historial_capital, mode='lines+markers', name='Balance',
                         line=dict(color=color_capital, width=3), marker=dict(size=6)), row=2, col=1)
fig.add_hline(y=100.0, line_dash="dash", line_color="black", row=2, col=1)

fig.update_layout(title=f'Backtest Operativo {symb} | Win Rate: {win_rate:.1f}%', height=900, template='plotly_white', hovermode='x unified', xaxis_rangeslider_visible=False)
fig.update_xaxes(rangebreaks=[dict(bounds=["sat", "mon"])])
fig.show()

-> Ejecutando Motor de Backtesting...
[+] Backtest Finalizado | Trades: 6 | Win Rate: 83.3% | Capital Final: $131.00
